# Comparing CometMirror and Torax

This notebook aims to show how you can run the SPARC PRD scenario in both CometMirror and Torax and then visualize the results for the two models.


## Running CometMirror for SPARC PRD
See `demos/comet_mirror.ipynb` for a more in-depth intro.

In [ ]:
%load_ext autoreload
%autoreload 2

from popsim.simulate import make_time_base, simulate
from popsim.simulators.comet_mirror.scenarios.sparc_prd import build_comet_mirror_config

# Initialize the simulator.
model, state, params = build_comet_mirror_config()

# Make a time base for all of our simulations.
time_base = make_time_base(t0=0.0, t1=5.0, dt=0.01)

cm_ds = simulate(
    module=model,
    time_base=time_base,
    initial_state=state,
    params=params,
)
cm_ds = cm_ds.assign_coords(rho=model.config.rho)  # TODO: make simulate automatically assign to xarray output.

## Running Torax for the SPARC PRD
To see the underlying configuration, you can click on `get_sim` below and hit `F12` if you are using VSCode.

In [ ]:
import os

from torax import simulation_app

import popsim
from popsim.simulators.torax.scenarios.sparc_prd import get_sim

os.environ["TORAX_QLKNN_MODEL_PATH"] = popsim.TORAX_QLKNN_MODEL_PATH
torax_ds = simulation_app.main(get_sim)

## Splicing the Datasets Together
Okay, now the idea is to put the simulation results for the two simulators into an `xr.Dataset` along a new dimension "simulation", which the PopsimGUI already knows how to handle. The code below does a somewhat unfortunate amount of data wrangling to get them together.

In [ ]:
import panel as pn
import xarray as xr

from popsim.gui import PopsimGUI

# Rename variables to match. Also, convert torax ne to 1e19.
torax_ds = torax_ds.rename({"temp_el": "Te", "temp_ion": "Ti", "ne": "ne", "rho_cell": "rho"})
torax_ds["ne"] = 10 * torax_ds.ne  # Convert from 1e20 to 1e19 like CM.
cm_ds = cm_ds.rename(
    {
        "output.profiles.electron_temp_profile": "Te",
        "output.profiles.ion_temp_profile": "Ti",
        "output.profiles.electron_density_profile": "ne",
        "rho": "rho",
    }
)

# Build the xr.Dataset.
vars_to_compare = ["Te", "Ti", "ne"]
simulation_dim = xr.DataArray(["torax", "comet_mirror"], dims=["simulation"])
ds = xr.concat(
    [torax_ds[vars_to_compare], cm_ds[vars_to_compare]],
    dim=simulation_dim,
)
ds = ds.dropna("time", how="any")  # Downsample to the lowest common denominator in time.

## Visualizing!

In [ ]:
# Visualize!
gui = PopsimGUI(ds, time_dim="time", rho_dim="rho", simulation_dim="simulation")
pn.serve(gui.build_view())